# Deep Generative Models - Assignment 2
## Masked Autoregressive Flow (MAF) & CycleGAN

**Student Name**: [Your Name]  
**Student ID**: [Your ID]  

This notebook contains complete implementations for:
- **Part 1**: Masked Autoregressive Flow (MAF) for density estimation and anomaly detection
- **Part 2**: CycleGAN for unpaired image-to-image translation

---

### Table of Contents
1. [MAF Implementation](#maf)
   - MADE (Masked Autoencoder)
   - MAF Blocks
   - Training & Generation
   - Anomaly Detection
   
2. [CycleGAN Implementation](#cyclegan)
   - Generator (ResNet-based)
   - Discriminator (PatchGAN)
   - Loss Functions
   - Training Loop
   
3. [Experiments & Results](#results)
   - MAF Training on Capsule Dataset
   - Anomaly Detection Evaluation
   - CycleGAN Training
   - Qualitative & Quantitative Analysis

---


## Part 1: Masked Autoregressive Flow (MAF) - Section 1

### Implementation Details

**Architecture Specifications:**
- **Input Dimensions**: 128 × 128 × 3 = 49,152
- **MADE Architecture**:
  - Input layer: 49,152 → 512
  - Hidden layer: 512 → 512  
  - Output layer: 512 → 98,304 (2× input for scale and translation parameters)
- **Number of MAF Blocks**: 7
- **Batch Size**: 3
- **Epochs**: 100
- **Learning Rate**: 0.0001
- **Optimizer**: Adam

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class MaskedLinear(nn.Module):
    def __init__(self, in_features, out_features, mask):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.register_buffer('mask', mask) 

    def forward(self, x):
        masked_weight = self.linear.weight * self.mask
        return F.linear(x, masked_weight, self.linear.bias)

def create_masks(input_dim, hidden_dims, output_dim):
    masks = []
    m_input = torch.arange(1, input_dim + 1)
    
    m_hidden = []
    for hidden_dim in hidden_dims:
        m_h = torch.randint(low=1, high=input_dim, size=(hidden_dim,))
        m_hidden.append(m_h)
    
    m_output = torch.arange(1, output_dim // 2 + 1).repeat(2)
    
    mask = (m_input.unsqueeze(1) <= m_hidden[0].unsqueeze(0)).float()
    masks.append(mask)
    
    for i in range(len(hidden_dims) - 1):
        mask = (m_hidden[i].unsqueeze(1) <= m_hidden[i+1].unsqueeze(0)).float()
        masks.append(mask)
    
    mask = (m_hidden[-1].unsqueeze(1) < m_output.unsqueeze(0)).float()
    masks.append(mask)
    
    return masks

class MADE(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        masks = create_masks(input_dim, hidden_dims, output_dim)
        
        self.layers = nn.ModuleList()
        dims = [input_dim] + hidden_dims + [output_dim]
        
        for i in range(len(dims) - 1):
            self.layers.append(MaskedLinear(dims[i], dims[i+1], masks[i]))
    
    def forward(self, x):
        batch_size = x.shape[0]
        x = x.view(batch_size, -1)
        
        for i, layer in enumerate(self.layers[:-1]):
            x = F.relu(layer(x))
        
        x = self.layers[-1](x)
        
        return x


In [ ]:
class MAFBlock(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 512]):
        super().__init__()
        self.input_dim = input_dim
        self.made = MADE(input_dim, hidden_dims, 2 * input_dim)

    def forward(self, x):
        batch_size = x.shape[0]
        x_flat = x.view(batch_size, -1)
        
        s_and_t = self.made(x_flat)
        s, t = s_and_t.chunk(2, dim=1)
        
        s = torch.sigmoid(s + 2.0)
        
        z = (x_flat - t) / (s + 1e-8)
        
        log_det_J = -torch.sum(torch.log(s + 1e-8), dim=1)
        
        return z, log_det_J

    def inverse(self, z):
        batch_size = z.shape[0]
        x = torch.zeros_like(z)
        
        for i in range(self.input_dim):
            s_and_t = self.made(x)
            s, t = s_and_t.chunk(2, dim=1)
            s = torch.sigmoid(s + 2.0)
            
            x[:, i] = s[:, i] * z[:, i] + t[:, i]
        
        return x


In [ ]:
class MAF(nn.Module):
    def __init__(self, input_dim, num_blocks=7, hidden_dims=[512, 512]):
        super().__init__()
        self.input_dim = input_dim
        self.blocks = nn.ModuleList([
            MAFBlock(input_dim, hidden_dims) for _ in range(num_blocks)
        ])
        
        self.register_buffer('base_mean', torch.zeros(input_dim))
        self.register_buffer('base_std', torch.ones(input_dim))

    def forward(self, x):
        batch_size = x.shape[0]
        x_flat = x.view(batch_size, -1)
        
        log_prob = torch.zeros(batch_size, device=x.device)
        
        z = x_flat
        for block in self.blocks:
            z, log_det_J = block(z)
            log_prob += log_det_J
        
        log_prob_base = -0.5 * (z ** 2 + np.log(2 * np.pi)).sum(dim=1)
        log_prob += log_prob_base
        
        return z, log_prob
    
    def calculate_nll(self, x):
        _, log_prob = self.forward(x)
        return -log_prob.mean()
        
    def generate(self, num_samples, device='cpu'):
        z = torch.randn(num_samples, self.input_dim, device=device)
        
        for block in reversed(self.blocks):
            z = block.inverse(z)
        
        return z


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
from tqdm import tqdm
import time
import matplotlib.pyplot as plt

class CapsuleDataset(Dataset):
    def __init__(self, root_dir, transform=None, img_size=128):
        self.root_dir = root_dir
        self.transform = transform
        self.img_size = img_size
        self.images = []
        
        for fname in os.listdir(root_dir):
            if fname.endswith(('.png', '.jpg', '.jpeg')):
                self.images.append(os.path.join(root_dir, fname))
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image

def train_maf(model, train_loader, num_epochs=100, lr=0.0001, device='cpu'):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch in pbar:
            batch = batch.to(device)
            
            optimizer.zero_grad()
            nll = model.calculate_nll(batch)
            
            nll.backward()
            optimizer.step()
            
            epoch_loss += nll.item()
            pbar.set_postfix({'NLL': nll.item()})
        
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        print(f'Epoch {epoch+1}, Average NLL: {avg_loss:.4f}')
    
    return losses

def generate_images_maf(model, num_images=5, img_size=128, device='cpu'):
    model.eval()
    
    start_time = time.time()
    
    with torch.no_grad():
        samples = model.generate(num_images, device=device)
        samples = samples.view(num_images, 3, img_size, img_size)
        samples = torch.clamp(samples, -1, 1)
        samples = (samples + 1) / 2
    
    generation_time = time.time() - start_time
    
    return samples, generation_time

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])


In [ ]:
lambda_A = 10.0
lambda_B = 10.0
lambda_identity = 0.5

def adversarial_loss(prediction, is_real):
    if is_real:
        target = torch.ones_like(prediction)
    else:
        target = torch.zeros_like(prediction)
    return F.mse_loss(prediction, target)

def cycle_consistency_loss(real_image, reconstructed_image):
    return F.l1_loss(reconstructed_image, real_image)

def identity_loss(generator, real_image):
    identity_image = generator(real_image)
    return F.l1_loss(identity_image, real_image)

def generator_loss(D, fake_image):
    pred_fake = D(fake_image)
    return adversarial_loss(pred_fake, True)

def discriminator_loss(D, real_image, fake_image):
    pred_real = D(real_image)
    pred_fake = D(fake_image.detach())
    
    loss_real = adversarial_loss(pred_real, True)
    loss_fake = adversarial_loss(pred_fake, False)
    
    return (loss_real + loss_fake) * 0.5


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_nc=3, ndf=64):
        super().__init__()
        
        model = [
            nn.Conv2d(input_nc, ndf, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        ]
        
        model += [
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True)
        ]
        
        model += [
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True)
        ]
        
        model += [
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=1, padding=1),
            nn.InstanceNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True)
        ]
        
        model += [
            nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=1)
        ]
        
        self.model = nn.Sequential(*model)
    
    def forward(self, x):
        return self.model(x)


In [ ]:
import random
from collections import deque

class ImagePool:
    def __init__(self, pool_size=50):
        self.pool_size = pool_size
        self.images = []
    
    def query(self, images):
        if self.pool_size == 0:
            return images
        
        return_images = []
        
        for image in images:
            image = image.unsqueeze(0)
            
            if len(self.images) < self.pool_size:
                self.images.append(image)
                return_images.append(image)
            else:
                if random.uniform(0, 1) > 0.5:
                    random_id = random.randint(0, self.pool_size - 1)
                    return_images.append(self.images[random_id].clone())
                    self.images[random_id] = image
                else:
                    return_images.append(image)
        
        return torch.cat(return_images, dim=0)


In [ ]:
def train_cyclegan(G_AB, G_BA, D_A, D_B, train_loader_A, train_loader_B, 
                   num_epochs=20, lr=0.0002, beta1=0.5, device='cpu'):
    
    G_AB = G_AB.to(device)
    G_BA = G_BA.to(device)
    D_A = D_A.to(device)
    D_B = D_B.to(device)
    
    optimizer_G = optim.Adam(
        list(G_AB.parameters()) + list(G_BA.parameters()),
        lr=lr, betas=(beta1, 0.999)
    )
    optimizer_D_A = optim.Adam(D_A.parameters(), lr=lr, betas=(beta1, 0.999))
    optimizer_D_B = optim.Adam(D_B.parameters(), lr=lr, betas=(beta1, 0.999))
    
    scheduler_G = optim.lr_scheduler.LambdaLR(
        optimizer_G, lr_lambda=lambda epoch: 1.0 - max(0, epoch - num_epochs // 2) / (num_epochs // 2)
    )
    scheduler_D_A = optim.lr_scheduler.LambdaLR(
        optimizer_D_A, lr_lambda=lambda epoch: 1.0 - max(0, epoch - num_epochs // 2) / (num_epochs // 2)
    )
    scheduler_D_B = optim.lr_scheduler.LambdaLR(
        optimizer_D_B, lr_lambda=lambda epoch: 1.0 - max(0, epoch - num_epochs // 2) / (num_epochs // 2)
    )
    
    fake_A_pool = ImagePool(pool_size=50)
    fake_B_pool = ImagePool(pool_size=50)
    
    history = {
        'G_loss': [],
        'D_A_loss': [],
        'D_B_loss': [],
        'cycle_loss': [],
        'identity_loss': []
    }
    
    for epoch in range(num_epochs):
        G_AB.train()
        G_BA.train()
        D_A.train()
        D_B.train()
        
        epoch_G_loss = 0
        epoch_D_A_loss = 0
        epoch_D_B_loss = 0
        epoch_cycle_loss = 0
        epoch_identity_loss = 0
        
        data_iter_A = iter(train_loader_A)
        data_iter_B = iter(train_loader_B)
        
        num_batches = min(len(train_loader_A), len(train_loader_B))
        pbar = tqdm(range(num_batches), desc=f'Epoch {epoch+1}/{num_epochs}')
        
        for i in pbar:
            try:
                real_A = next(data_iter_A).to(device)
                real_B = next(data_iter_B).to(device)
            except StopIteration:
                break
            
            batch_size = min(real_A.size(0), real_B.size(0))
            real_A = real_A[:batch_size]
            real_B = real_B[:batch_size]
            
            optimizer_G.zero_grad()
            
            loss_identity_A = identity_loss(G_BA, real_A) * lambda_A * lambda_identity
            loss_identity_B = identity_loss(G_AB, real_B) * lambda_B * lambda_identity
            loss_identity_total = loss_identity_A + loss_identity_B
            
            fake_B = G_AB(real_A)
            loss_GAN_AB = generator_loss(D_B, fake_B)
            
            fake_A = G_BA(real_B)
            loss_GAN_BA = generator_loss(D_A, fake_A)
            
            recovered_A = G_BA(fake_B)
            loss_cycle_A = cycle_consistency_loss(real_A, recovered_A) * lambda_A
            
            recovered_B = G_AB(fake_A)
            loss_cycle_B = cycle_consistency_loss(real_B, recovered_B) * lambda_B
            
            loss_cycle_total = loss_cycle_A + loss_cycle_B
            
            loss_G = loss_GAN_AB + loss_GAN_BA + loss_cycle_total + loss_identity_total
            
            loss_G.backward()
            optimizer_G.step()
            
            optimizer_D_A.zero_grad()
            
            fake_A_pooled = fake_A_pool.query(fake_A.detach())
            loss_D_A = discriminator_loss(D_A, real_A, fake_A_pooled)
            
            loss_D_A.backward()
            optimizer_D_A.step()
            
            optimizer_D_B.zero_grad()
            
            fake_B_pooled = fake_B_pool.query(fake_B.detach())
            loss_D_B = discriminator_loss(D_B, real_B, fake_B_pooled)
            
            loss_D_B.backward()
            optimizer_D_B.step()
            
            epoch_G_loss += loss_G.item()
            epoch_D_A_loss += loss_D_A.item()
            epoch_D_B_loss += loss_D_B.item()
            epoch_cycle_loss += loss_cycle_total.item()
            epoch_identity_loss += loss_identity_total.item()
            
            pbar.set_postfix({
                'G': f'{loss_G.item():.3f}',
                'D_A': f'{loss_D_A.item():.3f}',
                'D_B': f'{loss_D_B.item():.3f}'
            })
        
        scheduler_G.step()
        scheduler_D_A.step()
        scheduler_D_B.step()
        
        history['G_loss'].append(epoch_G_loss / num_batches)
        history['D_A_loss'].append(epoch_D_A_loss / num_batches)
        history['D_B_loss'].append(epoch_D_B_loss / num_batches)
        history['cycle_loss'].append(epoch_cycle_loss / num_batches)
        history['identity_loss'].append(epoch_identity_loss / num_batches)
        
        print(f'\nEpoch {epoch+1} - G: {history["G_loss"][-1]:.4f}, '
              f'D_A: {history["D_A_loss"][-1]:.4f}, D_B: {history["D_B_loss"][-1]:.4f}, '
              f'Cycle: {history["cycle_loss"][-1]:.4f}, Identity: {history["identity_loss"][-1]:.4f}')
    
    return history


In [ ]:
class ImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        
        for fname in os.listdir(root_dir):
            if fname.endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                self.images.append(os.path.join(root_dir, fname))
        
        print(f"Loaded {len(self.images)} images from {root_dir}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx % len(self.images)]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image

cyclegan_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])


In [ ]:
def visualize_samples(images, titles=None, figsize=(15, 5)):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    
    if n == 1:
        axes = [axes]
    
    for i, (img, ax) in enumerate(zip(images, axes)):
        if isinstance(img, torch.Tensor):
            img = img.cpu().detach()
            if img.dim() == 4:
                img = img[0]
            img = img.permute(1, 2, 0).numpy()
            img = (img + 1) / 2
            img = np.clip(img, 0, 1)
        
        ax.imshow(img)
        ax.axis('off')
        if titles and i < len(titles):
            ax.set_title(titles[i])
    
    plt.tight_layout()
    plt.show()

def test_cyclegan(G_AB, G_BA, test_loader_A, test_loader_B, device='cpu', num_samples=5):
    G_AB.eval()
    G_BA.eval()
    
    with torch.no_grad():
        real_A = next(iter(test_loader_A))[:num_samples].to(device)
        fake_B = G_AB(real_A)
        recovered_A = G_BA(fake_B)
        
        real_B = next(iter(test_loader_B))[:num_samples].to(device)
        fake_A = G_BA(real_B)
        recovered_B = G_AB(fake_A)
    
    for i in range(num_samples):
        visualize_samples(
            [real_A[i], fake_B[i], recovered_A[i]],
            titles=['Real A', 'Fake B', 'Recovered A']
        )
    
    for i in range(num_samples):
        visualize_samples(
            [real_B[i], fake_A[i], recovered_B[i]],
            titles=['Real B', 'Fake A', 'Recovered B']
        )

def plot_training_history(history):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    axes[0, 0].plot(history['G_loss'])
    axes[0, 0].set_title('Generator Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(history['D_A_loss'], label='D_A')
    axes[0, 1].plot(history['D_B_loss'], label='D_B')
    axes[0, 1].set_title('Discriminator Losses')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    axes[1, 0].plot(history['cycle_loss'])
    axes[1, 0].set_title('Cycle Consistency Loss')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(history['identity_loss'])
    axes[1, 1].set_title('Identity Loss')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

def calculate_anomaly_scores(model, test_loader, device='cpu'):
    model.eval()
    anomaly_scores = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Computing anomaly scores'):
            batch = batch.to(device)
            _, log_prob = model.forward(batch)
            nll = -log_prob
            anomaly_scores.extend(nll.cpu().numpy())
    
    return np.array(anomaly_scores)

def evaluate_anomaly_detection(normal_scores, anomaly_scores):
    y_true = np.concatenate([
        np.zeros(len(normal_scores)),
        np.ones(len(anomaly_scores))
    ])
    
    y_scores = np.concatenate([normal_scores, anomaly_scores])
    
    auroc = roc_auc_score(y_true, y_scores)
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    
    return auroc, fpr, tpr

def plot_roc_curve(fpr, tpr, auroc):
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC Curve (AUROC = {auroc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve for Anomaly Detection')
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_score_distributions(normal_scores, anomaly_scores):
    plt.figure(figsize=(10, 6))
    plt.hist(normal_scores, bins=50, alpha=0.6, label='Normal', density=True)
    plt.hist(anomaly_scores, bins=50, alpha=0.6, label='Anomaly', density=True)
    plt.xlabel('Anomaly Score (NLL)')
    plt.ylabel('Density')
    plt.title('Distribution of Anomaly Scores')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# Anomaly Detection using MAF

from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

def calculate_anomaly_scores(model, test_loader, device='cpu'):
    """
    Calculate anomaly scores for test images using MAF.
    Anomaly score = Negative Log Likelihood (NLL)
    Higher NLL indicates the image is less likely under the learned distribution
    """
    model.eval()
    anomaly_scores = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Computing anomaly scores'):
            batch = batch.to(device)
            _, log_prob = model.forward(batch)
            nll = -log_prob  # Negative log likelihood as anomaly score
            anomaly_scores.extend(nll.cpu().numpy())
    
    return np.array(anomaly_scores)

def evaluate_anomaly_detection(normal_scores, anomaly_scores):
    """
    Evaluate anomaly detection performance using AUROC
    
    Args:
        normal_scores: Anomaly scores for normal images (lower is better)
        anomaly_scores: Anomaly scores for anomalous images (higher is better)
    
    Returns:
        auroc: Area Under ROC Curve
        fpr: False Positive Rate
        tpr: True Positive Rate
    """
    # Create labels: 0 for normal, 1 for anomaly
    y_true = np.concatenate([
        np.zeros(len(normal_scores)),
        np.ones(len(anomaly_scores))
    ])
    
    # Combine scores
    y_scores = np.concatenate([normal_scores, anomaly_scores])
    
    # Calculate AUROC
    auroc = roc_auc_score(y_true, y_scores)
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    
    return auroc, fpr, tpr

def plot_roc_curve(fpr, tpr, auroc):
    """Plot ROC curve"""
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC Curve (AUROC = {auroc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve for Anomaly Detection')
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_score_distributions(normal_scores, anomaly_scores):
    """Plot distributions of anomaly scores"""
    plt.figure(figsize=(10, 6))
    plt.hist(normal_scores, bins=50, alpha=0.6, label='Normal', density=True)
    plt.hist(anomaly_scores, bins=50, alpha=0.6, label='Anomaly', density=True)
    plt.xlabel('Anomaly Score (NLL)')
    plt.ylabel('Density')
    plt.title('Distribution of Anomaly Scores')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# ==============================================================================
# PART 1: MAF Training and Evaluation Example
# ==============================================================================

"""
Example usage for MAF training on Capsule dataset

# 1. Download and prepare data
!wget https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420937454-1629951595/capsule.tar.xz
!tar -xf capsule.tar.xz

# 2. Setup dataset and dataloader
train_dataset = CapsuleDataset(
    root_dir='capsule/train/good',
    transform=transform,
    img_size=128
)

train_loader = DataLoader(
    train_dataset,
    batch_size=3,
    shuffle=True,
    num_workers=2
)

# 3. Initialize MAF model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = 128 * 128 * 3  # 49152
maf_model = MAF(
    input_dim=input_dim,
    num_blocks=7,
    hidden_dims=[512, 512]
)

print(f"Model parameters: {sum(p.numel() for p in maf_model.parameters()):,}")

# 4. Train the model
losses = train_maf(
    model=maf_model,
    train_loader=train_loader,
    num_epochs=100,
    lr=0.0001,
    device=device
)

# 5. Plot training losses
plt.figure(figsize=(10, 6))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Negative Log Likelihood')
plt.title('MAF Training Loss')
plt.grid(True)
plt.show()

# 6. Generate images (slow!)
print("Generating 5 images...")
generated_images, gen_time = generate_images_maf(
    model=maf_model,
    num_images=5,
    img_size=128,
    device=device
)

print(f"Generation time: {gen_time:.2f} seconds ({gen_time/5:.2f} sec per image)")

# 7. Visualize generated images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    img = generated_images[i].permute(1, 2, 0).cpu().numpy()
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()
"""


## Part 1: MAF - Section 2: Training and Generation

### Question Answers:

#### Q1: Why is generation slow in autoregressive models?

**Answer**: In autoregressive models like MAF, generation is slow because of the **sequential dependency** in the inverse transformation:

1. **Forward Pass (Training)**: Fast and parallelizable
   - Given input x, compute z = (x - t(x)) / s(x)
   - All dimensions can be computed in parallel since MADE can process the entire input at once
   
2. **Inverse Pass (Generation)**: Slow and sequential
   - To generate x from z, we need: x = s(x) · z + t(x)
   - **Problem**: Both s and t depend on x itself!
   - Must compute dimension by dimension:
     ```
     for i in 1 to D:
         x[i] = s[i](x[1:i-1]) · z[i] + t[i](x[1:i-1])
     ```
   - Each dimension requires a full forward pass through MADE
   - For 49,152 dimensions, this means 49,152 sequential MADE evaluations!

**Time Complexity**:
- Forward: O(1) - single pass
- Inverse: O(D) - D sequential passes where D is dimensionality

This is why generating 5 images can take several minutes!

---

#### Q2: How does Inverse Autoregressive Flow (IAF) solve this problem?

**Answer**: IAF reverses the direction of the autoregressive dependency:

**IAF Forward (Generation)**:
- z → x: **x = s(z) · z + t(z)**  [FAST - parallel]
- Parameters depend on z, not x
- All dimensions computed in one pass

**IAF Inverse (Training)**:
- x → z: **z = (x - t(z)) / s(z)**  [SLOW - sequential]
- Need to solve for z iteratively

**Comparison**:
| Model | Training | Generation |
|-------|----------|------------|
| MAF   | Fast ✅   | Slow ❌     |
| IAF   | Slow ❌   | Fast ✅     |

**When to use**:
- **MAF**: When you need fast training (e.g., density estimation, anomaly detection)
- **IAF**: When you need fast generation (e.g., VAE posterior, generative models)

---

In [ ]:
# ==============================================================================
# PART 2: Anomaly Detection with MAF
# ==============================================================================

"""
Example usage for anomaly detection with trained MAF

# 1. Prepare test datasets (normal and anomaly)
test_normal_dataset = CapsuleDataset(
    root_dir='capsule/test/good',
    transform=transform,
    img_size=128
)

test_anomaly_dataset = CapsuleDataset(
    root_dir='capsule/test/crack',  # or other defect types
    transform=transform,
    img_size=128
)

test_normal_loader = DataLoader(test_normal_dataset, batch_size=8, shuffle=False)
test_anomaly_loader = DataLoader(test_anomaly_dataset, batch_size=8, shuffle=False)

# 2. Calculate anomaly scores
print("Calculating anomaly scores for normal test images...")
normal_scores = calculate_anomaly_scores(maf_model, test_normal_loader, device)

print("Calculating anomaly scores for anomalous test images...")
anomaly_scores = calculate_anomaly_scores(maf_model, test_anomaly_loader, device)

# 3. Evaluate performance
auroc, fpr, tpr = evaluate_anomaly_detection(normal_scores, anomaly_scores)
print(f"AUROC: {auroc:.4f}")

# 4. Plot results
plot_roc_curve(fpr, tpr, auroc)
plot_score_distributions(normal_scores, anomaly_scores)

# 5. Show some examples
print(f"Normal images - Mean score: {normal_scores.mean():.4f}, Std: {normal_scores.std():.4f}")
print(f"Anomaly images - Mean score: {anomaly_scores.mean():.4f}, Std: {anomaly_scores.std():.4f}")
"""


### Section 2: Model Architecture and Training Process

#### Q1: Why does CycleGAN use two separate generators instead of one? Explain from theoretical and practical perspectives.

**Answer**:

**Theoretical Reasons**:

**1. Asymmetric Transformations**:
- X → Y transformation is fundamentally different from Y → X
- Example: Horse → Zebra (add stripes) vs Zebra → Horse (remove stripes)
- Different operations require different learned features
- A single shared generator can't capture both directions effectively

**2. Domain-Specific Features**:
```
G_AB: Learns horse features → zebra features
G_BA: Learns zebra features → horse features
```
- Each domain has unique characteristics
- Separate networks can specialize for each direction
- More expressive and flexible

**3. Cycle Consistency Requirement**:
```
x → G(x) → F(G(x)) ≈ x
```
- Need F and G to be approximate inverses
- With one generator: G must be its own inverse (very restrictive!)
- With two: G and F can be different functions that compose to identity

---

**Practical Reasons**:

**1. Training Stability**:
- Each generator has its own learning dynamics
- Separate optimizers and learning rates possible
- One direction can converge faster without blocking the other

**2. Capacity and Flexibility**:
- Different domains may need different model capacities
- More parameters = more expressive power

**3. Real-World Usage**:
- Users often care about only one direction
- Can deploy only G_AB without G_BA if only need X → Y
- Modular design

---

#### Q2: Compare the generator architecture used in CycleGAN with pix2pix.

**Answer**:

**Pix2pix Generator: U-Net Architecture**

```
Encoder:         Decoder:
Input            Output
  ↓               ↑
C64  ──────────→ C64 + ReLU + UpSample
  ↓               ↑
C128 ──────────→ C128 + ReLU + UpSample
  ↓               ↑
C256 ──────────→ C256 + ReLU + UpSample

Key: Skip connections (──→) at each level
```

**Features**:
- **Skip connections** from encoder to decoder
- Direct low-level information flow
- Preserves spatial details
- Good for pixel-aligned tasks

---

**CycleGAN Generator: ResNet Architecture**

```
Input
  ↓
C7S1-64 (Initial Conv)
  ↓
D128 (Downsampling)
  ↓
D256 (Downsampling)
  ↓
R256 × 9 (9 Residual Blocks)
  ↓
U128 (Upsampling)
  ↓
U64 (Upsampling)
  ↓
C7S1-3 (Output Conv + Tanh)

Key: No skip connections between down/up sampling
```

---

**Why Are They Different?**

**1. Task Difference**:

**Pix2pix (Paired, Aligned)**:
- Needs to preserve **exact spatial layout**
- Pixel-to-pixel correspondence crucial
- Skip connections maintain alignment

**CycleGAN (Unpaired, Unaligned)**:
- Needs to preserve **semantic content**, not pixels
- No pixel alignment required
- Must learn higher-level structure

**2. Information Bottleneck**:

**U-Net (Pix2pix)**:
- Skip connections → Low-level features bypass bottleneck
- Good: Maintains spatial precision

**ResNet (CycleGAN)**:
- No skips → All info goes through bottleneck
- Forces model to **understand** content
- Good: Semantic transformation

---

#### Q3: Why use PatchGAN discriminator?

**Answer**:

**PatchGAN Concept**:

Instead of classifying the **entire image** as real/fake, PatchGAN outputs an **N×N matrix** where each element classifies a **local patch** (e.g., 70×70 pixels).

**Why Use PatchGAN?**

**1. Captures Local Texture and Style**:
- High-frequency details (edges, textures) are local
- Each patch discriminator focuses on small regions
- Better at detecting **texture-level artifacts**

**2. Fewer Parameters**:
```
Full Discriminator (256×256 → 1): ~2.7M parameters
PatchGAN (256×256 → 30×30): ~0.5M parameters ✅
```

**3. Applies to Any Image Size**:
- Fully convolutional → Works on any size
- Train on 256×256, test on 512×512 ✅

**4. Better for High-Resolution Details**:
- **Global** (shape, pose): Handled by cycle consistency
- **Local** (stripe texture): Handled by PatchGAN ✅

---

#### Q4: Why use an image history buffer instead of only the latest generated images?

**Answer**:

**The Problem: Model Oscillation**

Without image buffer, discriminator only trains on **most recent** generator outputs:

```
Iteration 1:
  G generates: Fake_1
  D trains on: Fake_1

Iteration 2:
  G adapts: Generates Fake_2
  D trains on: Fake_2 (forgets about Fake_1!)
  
Iteration 3:
  G reverts: Generates Fake_1 again
  
→ Infinite oscillation! ❌
```

**The Solution: Image History Buffer**

Store the last 50 generated images and randomly sample from them:

**Benefits**:

**1. Prevents Forgetting**:
- D maintains **memory** of past generator outputs
- G can't exploit forgotten patterns

**2. Reduces Oscillation**:
```
Without Buffer: Style_1 → Style_2 → Style_1 → Style_2 ...
With Buffer: Style_1 → Style_2 → Style_3 → Style_4 ... ✅
```

**3. Stabilizes Training**:
- Smoother loss curves
- Better convergence

**4. Covers Distribution Better**:
- Discriminator sees **diverse** samples
- Better coverage of fake image space

**Why 50 Images?**
- Empirically optimal balance
- Balances memory vs stability

---

### Section 3: Model Limitations

#### Q1: What does it mean that CycleGAN has poor performance on tasks requiring large geometric changes? Why does cycle consistency prevent learning such transformations?

**Answer**:

**The Limitation Explained**:

CycleGAN struggles with transformations that require **significant structural or geometric changes**:

**Examples of Difficult Tasks**:
```
❌ Dog → Cat (different body structure)
❌ Car → Bicycle (different number of wheels)
❌ Face → Cartoon face (exaggerated proportions)
❌ Standing person → Sitting person (pose change)
❌ Apple → Banana (different shape)
```

**Examples of Easy Tasks**:
```
✅ Horse → Zebra (same shape, different texture)
✅ Summer → Winter (same scene, different color/texture)
✅ Day → Night (same structure, different lighting)
✅ Photo → Painting style (same content, different style)
```

---

**Why Cycle Consistency Prevents Geometric Changes**:

**Mathematical Constraint**:
```
Cycle consistency requires:
F(G(x)) ≈ x  AND  G(F(y)) ≈ y
```

**Problem**: This constraint is **too strong** for geometric changes!

**Detailed Explanation**:

**Step 1: Forward Translation**
```
Input: Dog image (4 legs, specific pose)
G(dog) → Cat image

To look realistic:
- Must have cat features (whiskers, cat face)
- May need different body proportions
- Different leg positions
```

**Step 2: Cycle Back**
```
Cat image → F(cat) → Should recover original dog

Problem:
- Cat structure is different from dog
- How to remember exact dog pose?
- Cat proportions don't match dog proportions
- Information about original dog structure is LOST!
```

**The Paradox**:
```
To make realistic cat: Need to change structure significantly
To recover original dog: Need to preserve structure exactly

These two requirements CONFLICT! ❌
```

---

**Concrete Example: Dog → Cat**

**Without Geometric Change** (What CycleGAN does):
```
Input:  Dog with brown fur, standing
Step 1: G(dog) = "Dog-like cat" (cat texture on dog structure)
Step 2: F(G(dog)) = Original dog ✅ (cycle closes)

Result: Cat with dog proportions (unrealistic cat!)
```

**With Geometric Change** (What we want but can't have):
```
Input:  Dog with specific pose
Step 1: G(dog) = Real cat (different structure, realistic)
Step 2: F(G(dog)) = ??? (can't recover exact dog pose!)

Result: F(G(dog)) ≠ original dog ❌ (cycle broken)
       High cycle consistency loss
       Training fails!
```

---

**Why Information is Lost**:

**1. Structural Information Bottleneck**:
```
Dog image contains:
- Specific leg positions
- Tail angle
- Ear shape
- Body proportions

If G changes these to cat proportions:
→ Information is DISCARDED
→ F cannot reconstruct it
```

**2. Bijection Requirement**:

Cycle consistency implicitly requires:
```
G ∘ F ≈ identity
F ∘ G ≈ identity
```

This means G and F should be **approximate inverses**.

**For texture changes**: Invertible ✅
```
Add stripes (horse→zebra) ←→ Remove stripes (zebra→horse)
```

**For structure changes**: NOT invertible ❌
```
Lengthen legs (dog→cat) ←→ ??? (can't uniquely reverse)
```

---

**Mathematical Insight**:

**L1 Cycle Loss**:
```
L_cyc = ||F(G(x)) - x||₁
```

To minimize this loss:
```
F(G(x)) must be pixel-close to x
→ G(x) cannot differ too much structurally from x
→ Only allows "style transfer" not "shape transfer"
```

**The Constraint**:
- Cycle loss **penalizes** large structural changes
- Model learns to preserve structure at all costs
- Only surface-level changes (texture, color) are allowed

---

**Analogy**:

**Translating Languages**:

**Texture change (Easy)**:
```
English: "The cat sat on the mat"
Spanish: "El gato se sentó en la alfombra"

Can translate back perfectly ✅
Same structure, different words
```

**Structure change (Hard)**:
```
English: "The cat sat on the mat" (Subject-Verb-Object)
Japanese: "猫はマットの上に座った" (Subject-Object-Verb)

If I only remember Spanish structure:
→ Can't perfectly reconstruct English word order ❌
```

---

**What CycleGAN Actually Learns**:

**Horse ↔ Zebra** (Successful):
```
G: Change texture (add stripes)
F: Change texture (remove stripes)
Structure: Preserved perfectly ✅
```

**Dog ↔ Cat** (Fails):
```
G: Try to change structure → High cycle loss ❌
   Compromise: Only change texture
F: Reverse texture change
Result: "Cat-textured dogs" and "Dog-textured cats"
```

---

**Experimental Evidence**:

**Cycle Consistency Weight λ**:

```
λ = 0 (no cycle loss):
- Allows geometric changes
- But: mode collapse, meaningless translations ❌

λ = 10 (standard):
- Stable training ✅
- Only texture/style changes ✅
- No geometric changes ❌

λ = 100 (very high):
- Almost no translation at all ❌
```

**Trade-off is unavoidable**!

---

**Solutions and Alternatives**:

**1. Use Paired Data** (if available):
- Pix2pix can learn geometric changes
- But requires expensive paired dataset

**2. Conditional CycleGAN**:
- Add explicit geometry control
- E.g., keypoint supervision

**3. Disentangled Representations**:
- Separate content (structure) and style (texture)
- MUNIT, DRIT

**4. Weaker Consistency**:
- Relaxed cycle consistency
- Allow some information loss

**5. 3D-aware models**:
- Explicitly model 3D structure
- Then only change texture

---

**Conclusion**:

**Cycle consistency** is both a blessing and a curse:
- ✅ Enables **unsupervised** learning without paired data
- ✅ Prevents **mode collapse** and meaningless mappings  
- ❌ Restricts to **structure-preserving** transformations
- ❌ Cannot handle **large geometric changes**

**Best use cases**: Tasks where style changes but structure remains (texture, color, lighting, artistic style)

**Avoid**: Tasks requiring shape/geometry changes (pose, object class, proportions)

---

## Part 1: MAF - Section 3: Anomaly Detection

### Question Answers:

#### Q1: Why don't we use accuracy as an evaluation metric for anomaly detection?

**Answer**: Accuracy is misleading in anomaly detection due to **severe class imbalance**:

**Example Scenario**:
- Normal samples: 95%
- Anomalies: 5%

A naive model that predicts "normal" for everything achieves **95% accuracy** but detects **0% of anomalies**! This is useless for anomaly detection.

**Why This Happens**:
- Anomalies are rare by definition
- The cost of missing an anomaly (false negative) is usually much higher than a false alarm (false positive)
- Accuracy treats all errors equally, ignoring this asymmetry

**Better Metrics**:
- **AUROC (Area Under ROC Curve)**: Measures performance across all thresholds
- **Precision-Recall AUC**: Better for highly imbalanced data
- **F1-Score**: Balances precision and recall
- **True Positive Rate at fixed False Positive Rate**: Domain-specific requirements

---

#### Q2: What is the concept of anomaly score?

**Answer**: An **anomaly score** is a scalar value that quantifies how "unusual" or "abnormal" a data point is compared to the normal distribution.

**Key Properties**:
- **Higher score → More anomalous**
- **Lower score → More normal**
- Provides a ranking rather than binary classification
- Allows flexible threshold selection based on use case

**Common Anomaly Scores**:

1. **Reconstruction Error** (VAE, Autoencoder):
   ```
   score = ||x - x_reconstructed||
   ```
   - Normal samples reconstruct well (low error)
   - Anomalies reconstruct poorly (high error)

2. **Negative Log-Likelihood** (Normalizing Flows, GMM):
   ```
   score = -log p(x)
   ```
   - Normal samples have high probability (low NLL)
   - Anomalies have low probability (high NLL)

3. **Distance-based** (One-Class SVM, Isolation Forest):
   ```
   score = distance_to_decision_boundary(x)
   ```

**Threshold Selection**:
- Set based on desired false positive rate
- Domain-specific requirements (e.g., medical: minimize false negatives)

---

#### Q3: How can normalizing flows be used for anomaly detection?

**Answer**: Normalizing flows are **excellent** for anomaly detection because they learn the **exact probability density** p(x).

**Method**:
1. **Training**: Learn the distribution of normal data
   - Train flow model on normal samples only
   - Model learns p(x) for normal distribution

2. **Anomaly Scoring**: Use negative log-likelihood
   ```python
   score(x) = -log p(x)
   ```
   - Normal samples: High p(x) → Low score
   - Anomalies: Low p(x) → High score

3. **Detection**: Threshold the scores
   ```python
   is_anomaly = score(x) > threshold
   ```

**Advantages**:
✅ **Exact density estimation** (unlike VAE which has intractable likelihood)
✅ **Principled probabilistic framework**
✅ **No reconstruction needed** (direct likelihood computation)
✅ **Works well for high-dimensional data**

**Why It Works**:
- Normal data lies in high-density regions of the learned distribution
- Anomalies lie in low-density regions (out-of-distribution)
- The likelihood directly measures "typicality"

**Practical Considerations**:
- Requires sufficient normal training data
- Sensitive to distribution shift
- Computational cost for high-dimensional data

---

#### Q4: Can we use normalizing flows for anomaly detection in the same way as VAE reconstruction error?

**Answer**: **Not exactly** - the approaches are fundamentally different, but both can work:

**VAE Approach (Reconstruction-Based)**:
```python
# Training: Learn encoder and decoder
x → encoder → z → decoder → x_reconstructed
loss = reconstruction_error + KL_divergence

# Testing: Measure reconstruction error
anomaly_score = ||x - x_reconstructed||²
```

**Why it works for VAE**:
- Normal samples are well-reconstructed (low error)
- Anomalies are poorly reconstructed (high error)
- Relies on the bottleneck: anomalies can't be encoded/decoded well

**Normalizing Flow Approach (Likelihood-Based)**:
```python
# Training: Learn exact density
x → flow → z (with log|det J|)
loss = -log p(x)

# Testing: Compute likelihood
anomaly_score = -log p(x)
```

**Why it works for Flows**:
- Direct density estimation, no reconstruction
- Anomalies have low probability under learned distribution

**Can we use reconstruction with Flows?**

**Theoretically YES**, but it's not the standard approach:
```python
# Forward: x → z
z, log_prob = flow.forward(x)

# Inverse: z → x_reconstructed
x_reconstructed = flow.inverse(z)

# Anomaly score
score = ||x - x_reconstructed||²
```

**Problems with this approach**:
❌ **Should be perfect reconstruction** (flows are bijective!)
❌ Only fails due to numerical errors, not semantic anomalies
❌ Doesn't leverage the main advantage of flows (exact likelihood)
❌ Much slower (requires expensive inverse pass)

**Conclusion**:
- **VAE**: Use reconstruction error (likelihood intractable)
- **Flows**: Use negative log-likelihood (exact and principled)
- Both work, but they exploit different properties of the models

**Best Practice**:
Stick to likelihood-based detection for normalizing flows - it's what they're designed for!

---

In [ ]:
# ==============================================================================
# PART 3: CycleGAN Training Example
# ==============================================================================

"""
Example usage for CycleGAN training

# 1. Download dataset (example: horse2zebra)
# You can use: apple2orange, summer2winter_yosemite, monet2photo, etc.
!wget https://people.eecs.berkeley.edu/~taesung_park/CycleGAN/datasets/horse2zebra.zip
!unzip horse2zebra.zip

# 2. Setup datasets
train_dataset_A = ImageDataset(
    root_dir='horse2zebra/trainA',
    transform=cyclegan_transform
)

train_dataset_B = ImageDataset(
    root_dir='horse2zebra/trainB',
    transform=cyclegan_transform
)

test_dataset_A = ImageDataset(
    root_dir='horse2zebra/testA',
    transform=cyclegan_transform
)

test_dataset_B = ImageDataset(
    root_dir='horse2zebra/testB',
    transform=cyclegan_transform
)

train_loader_A = DataLoader(train_dataset_A, batch_size=1, shuffle=True, num_workers=2)
train_loader_B = DataLoader(train_dataset_B, batch_size=1, shuffle=True, num_workers=2)
test_loader_A = DataLoader(test_dataset_A, batch_size=5, shuffle=False)
test_loader_B = DataLoader(test_dataset_B, batch_size=5, shuffle=False)

# 3. Visualize some samples
print("Sample images from domain A:")
sample_A = next(iter(test_loader_A))
visualize_samples(sample_A[:5])

print("Sample images from domain B:")
sample_B = next(iter(test_loader_B))
visualize_samples(sample_B[:5])

# 4. Initialize models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

G_AB = Generator(input_nc=3, output_nc=3, ngf=64, num_residual_blocks=9)
G_BA = Generator(input_nc=3, output_nc=3, ngf=64, num_residual_blocks=9)
D_A = Discriminator(input_nc=3, ndf=64)
D_B = Discriminator(input_nc=3, ndf=64)

print(f"G_AB parameters: {sum(p.numel() for p in G_AB.parameters()):,}")
print(f"G_BA parameters: {sum(p.numel() for p in G_BA.parameters()):,}")
print(f"D_A parameters: {sum(p.numel() for p in D_A.parameters()):,}")
print(f"D_B parameters: {sum(p.numel() for p in D_B.parameters()):,}")

# 5. Train the model
history = train_cyclegan(
    G_AB=G_AB,
    G_BA=G_BA,
    D_A=D_A,
    D_B=D_B,
    train_loader_A=train_loader_A,
    train_loader_B=train_loader_B,
    num_epochs=20,
    lr=0.0002,
    beta1=0.5,
    device=device
)

# 6. Plot training history
plot_training_history(history)

# 7. Test and visualize results at different epochs
print("\\nTesting at epoch 1 (early)...")
# Load checkpoint from epoch 1 and test
# test_cyclegan(G_AB, G_BA, test_loader_A, test_loader_B, device)

print("\\nTesting at epoch 10 (mid)...")
# Load checkpoint from epoch 10 and test
# test_cyclegan(G_AB, G_BA, test_loader_A, test_loader_B, device)

print("\\nTesting at epoch 20 (final)...")
test_cyclegan(G_AB, G_BA, test_loader_A, test_loader_B, device, num_samples=5)

# 8. Save models
torch.save(G_AB.state_dict(), 'G_AB_final.pth')
torch.save(G_BA.state_dict(), 'G_BA_final.pth')
torch.save(D_A.state_dict(), 'D_A_final.pth')
torch.save(D_B.state_dict(), 'D_B_final.pth')
"""
